# 02 Price model (cities A and B)
Split → baseline → features → models → choose on validation → test → gold.

Owner: P3. Logic is in `src/airbnb/` (`features.py`, `model.py`, `gold.py`).

The model must work on a new city, so it uses no neighbourhood names, no city name and no city prices.

In [0]:
import importlib, os, sys
import pandas as pd
from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../src"))
from airbnb import config, features, model, gold
for m in (config, features, model, gold):
    importlib.reload(m)

A, B = config.TRAIN_CITIES
silver = spark.table(config.LISTINGS_TABLE).filter(F.col("city").isin(A, B))
missing = features.REQUIRED_SILVER - set(silver.columns)
assert not missing, f"silver is missing {missing}: re-run 00_run_pipeline"

## 1. Split
20% test, 20% validation, 60% train (hash of the id).

In scope: has a price, not a price outlier, not a long stay. The rest stay in gold with an `exclusion_reason`.

In [0]:
df = model.add_split(silver)
dev = df.filter("in_scope AND split != 'test'")   # train + validation

display(df.groupBy("city", "split").agg(F.count("*").alias("listings"), F.sum(F.col("in_scope").cast("int")).alias("in_scope")).orderBy("city", "split"))
display(df.filter(~F.col("in_scope")).groupBy("city", "exclusion_reason").count().orderBy("city", "exclusion_reason"))

## 2. Baseline and features
Baseline: median price per `accommodates` and `room_type`.

| Feature | Why |
|---|---|
| `room_type`, `property_group` | what is rented |
| `accommodates`, `bedrooms`, `bathrooms`, `amenities_count` | size |
| `minimum_nights` (max 30) | stay length |
| `number_of_reviews`, `has_reviews`, `rating` | review history |
| `calculated_host_listings_count`, `host_tenure_months` | host type and experience |
| `dist_to_centre_km` | distance to the city centre (set in `config.py`) |
| `density_pct` | how crowded the area is, within the city |

Left out: neighbourhood names, `availability_365` (tested below), anything built from the price.

In [0]:
stats = features.fit_stats(dev)
df = (model.add_baseline(dev, features.make_features(df, stats))
      .withColumn("label", F.when(F.col("in_scope"), F.log("price"))))
scope = df.filter("in_scope")
train_df, val_df, test_df = (scope.filter(f"split = '{s}'") for s in ("train", "val", "test"))
dev_df = scope.filter("split != 'test'")
print("train:", train_df.count(), "| validation:", val_df.count(), "| test:", test_df.count())

## 3. Models
- Linear regression
- Random forest
- Random forest + `availability_365`
- Random forest + neighbourhood names (experiment: shows what we lose by not using them)

In [0]:
specs = model.candidates(dev_df)
models = {name: model.build(*spec).fit(train_df) for name, spec in specs.items()}

## 4. Choose on validation
MAE in AUD. Lowest wins. No neighbourhood model; availability must be 2% better.

In [0]:
val = model.evaluate_all(models, val_df)
display(val.pivot(index="model", columns="city", values="mae")[["all", A, B]].round(1).reset_index())
FINAL = model.choose_final(val)
print("Final model:", FINAL)

## 5. Test error
Test rows are used once. Final model and baseline learn from train + validation.

In [0]:
final = model.build(*specs[FINAL]).fit(dev_df)
portability = model.build(*specs[model.NB]).fit(dev_df)

results = (model.evaluate(test_df, "baseline_pred", "baseline")
           .unionByName(model.evaluate(model.scored(final, test_df), "pred", FINAL))
           .unionByName(model.evaluate(model.scored(portability, test_df), "pred", model.NB)))
table = results.toPandas()
for metric in ("mae", "medae"):
    display(table.pivot(index="model", columns="city", values=metric).reindex(["baseline", FINAL, model.NB])[["all", A, B]].round(1).reset_index())

## 6. Drivers and errors
Bias > 0 means the model over-prices.

In [0]:
test_scored = model.scored(final, test_df)

if hasattr(final.stages[-1], "featureImportances"):
    names = [c.replace("_idx", "") for c in final.stages[-2].getInputCols()]
    display(pd.DataFrame({"feature": names, "importance": final.stages[-1].featureImportances.toArray()}).sort_values("importance", ascending=False))

display(test_scored.groupBy("city", "room_type").agg(F.avg(F.abs(F.col("pred") - F.col("price"))).alias("mae"),
                                                     F.avg(F.col("pred") - F.col("price")).alias("bias"), F.count("*").alias("n")).orderBy("city", "room_type"))

## 7. Save
Gold tables: `price_predictions`, `model_metrics`, `city_summary`. Model numbers use `split = 'test' AND in_scope`.

The model and `feature_stats.json` go to the Volume. For a new city: `fitted, stats = model.load()`.

In [0]:
gold.write_predictions(model.scored(final, df))
gold.write_metrics(results, FINAL)
display(gold.write_city_summary(spark))
print(f"saved '{FINAL}' to {model.save(final, stats)}")